In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split
import torch.nn.functional as F
import numpy as np
import gc
from tqdm import tqdm

import matplotlib.pyplot as plt
import os
import seaborn as sns
from IPython.display import clear_output

In [ ]:
BATCH_SIZE = 8
NUM_WORKERS = 2
PYTORCH_THREADS = 2
LEARNING_RATE = 0.0007065405798027769
NUM_EPOCHS = 150
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(PYTORCH_THREADS)

print(f"Device: {DEVICE}")

print(f"Threads capped at: {PYTORCH_THREADS}")
print(f"Batch Size: {BATCH_SIZE}")

In [ ]:
# --- CONFIGURATION ---
INPUT_DIR = '/kaggle/input/datasets/itapita/lucidrf-unet-final'
TRAIN_X_PATH = os.path.join(INPUT_DIR, 'X_train2.npy')
TRAIN_Y_PATH = os.path.join(INPUT_DIR, 'Y_train2.npy')
VAL_X_PATH = os.path.join(INPUT_DIR, 'X_val2.npy')
VAL_Y_PATH = os.path.join(INPUT_DIR, 'Y_val2.npy')

checkpoint_path = '/kaggle/working/lucidrf_unet_checkpoint.pth'
log_file = "/kaggle/working/training_logs.txt"
train_fig = "/kaggle/working/training_plot.png"
best_model_fig = "/kaggle/working/best_model_plot.png"

In [ ]:
class DoubleConv(nn.Module):
    """
    (Conv1d => BN => ReLU) * 2
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class LucidRF_UNet(nn.Module):
    """
    1D U-Net for RF Signal Denoising.
    """
    def __init__(self, n_channels=2, n_classes=2, base=16):
        super(LucidRF_UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes

        # --- Encoder ---
        self.inc = DoubleConv(n_channels, base)
        self.down1 = nn.Sequential(nn.MaxPool1d(2), DoubleConv(base, base*2))
        self.down2 = nn.Sequential(nn.MaxPool1d(2), DoubleConv(base*2, base*4))
        self.down3 = nn.Sequential(nn.MaxPool1d(2), DoubleConv(base*4, base*8))
        self.down4 = nn.Sequential(nn.MaxPool1d(2), DoubleConv(base*8, base*16))

        # --- Decoder ---
        self.up1 = nn.ConvTranspose1d(base*16, base*8, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(base*16, base*8)
        
        self.up2 = nn.ConvTranspose1d(base*8, base*4, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(base*8, base*4)
        
        self.up3 = nn.ConvTranspose1d(base*4, base*2, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(base*4, base*2)
        
        self.up4 = nn.ConvTranspose1d(base*2, base, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv(base*2, base)

        # --- Output ---
        self.outc = nn.Conv1d(base, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5)
        x = torch.cat([x4, x], dim=1) # Skip connection
        x = self.conv_up1(x)

        x = self.up2(x)
        x = torch.cat([x3, x], dim=1) # Skip connection
        x = self.conv_up2(x)

        x = self.up3(x)
        x = torch.cat([x2, x], dim=1) # Skip connection
        x = self.conv_up3(x)

        x = self.up4(x)
        x = torch.cat([x1, x], dim=1) # Skip connection
        x = self.conv_up4(x)

        logits = self.outc(x)
        return logits

In [ ]:
class LucidRFDataset(Dataset):
    """
    PyTorch Dataset for LucidRF Denoising Task.
    
    Reads pre-split .npy tensors.
    Expected Input Shape: (N, 2, 40960) float32
    """
    def __init__(self, x_path, y_path):
        """
        Args:
            x_path (str): Path to the Noisy input .npy file
            y_path (str): Path to the Clean target .npy file
        """
        # Load data. Using mmap_mode='r' is safer for large files, 
        self.X_data = np.load(x_path, mmap_mode='r')
        self.Y_data = np.load(y_path, mmap_mode='r')
        
        # Verify alignment
        assert self.X_data.shape[0] == self.Y_data.shape[0], \
            f"Size Mismatch! X: {self.X_data.shape}, Y: {self.Y_data.shape}"


    def __len__(self):
        return self.X_data.shape[0]

    def __getitem__(self, idx):
        # Get raw samples
        # Shape expected: (2, 40960) or (40960,) complex depending on your save format
        x_sample = np.array(self.X_data[idx])
        y_sample = np.array(self.Y_data[idx])

        x_tensor = torch.from_numpy(x_sample).float()
        y_tensor = torch.from_numpy(y_sample).float()

        return x_tensor, y_tensor

In [ ]:
class EarlyStopping:
    """
    Stops training if validation loss doesn't improve after a given patience.
    """
    def __init__(self, patience=5, min_delta=1e-9):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < (self.best_loss - self.min_delta):
            self.best_loss = val_loss
            self.counter = 0  # Reset counter
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PhaseAwareLoss(nn.Module):
    def __init__(self, phase_weight=0.5):
        """
        Combines standard MSE with a Cosine Similarity penalty to enforce 
        phase alignment for RF constellations (like QPSK).
        """
        super().__init__()
        self.mse = nn.MSELoss()
        self.phase_weight = phase_weight

    def forward(self, y_pred, y_true):
        # Standard Amplitude/Waveform Loss (Maintains volume stability)
        mse_loss = self.mse(y_pred, y_true)
        
        # Phase Alignment (Cosine Similarity)
        # y_pred and y_true are shape (Batch, 2, L) where dim=1 is the I/Q channel
        # Cosine similarity evaluates to 1 if perfectly aligned, -1 if opposite.
        cos_sim = F.cosine_similarity(y_pred, y_true, dim=1)
        
        # We want to minimize the difference from perfect alignment (1.0)
        # Taking the mean across the batch and sequence length
        phase_penalty = torch.mean(1.0 - cos_sim)
        
        # Combined Objective
        total_loss = mse_loss + (self.phase_weight * phase_penalty)
        
        return total_loss

In [ ]:
def setup_plotting_style():
    """
    Configures the global plotting style for the entire project.
    Call this once at the start of your notebook.
    """
    # Set the theme (Seaborn handles matplotlib under the hood)
    sns.set_theme(
        style="whitegrid", 
        palette="deep",
        font_scale=1.2  
    )
    
    plt.rcParams['figure.figsize'] = (10, 6)  # Default size
    plt.rcParams['savefig.dpi'] = 300         # High-res saving
    
    print("Plotting style configured")

setup_plotting_style()


In [ ]:
def plot_training_live_history(history):
    """
    Updates the Loss and SINR curves in real-time.
    """
    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot Loss
    ax1.plot(history['train_loss'], label='Train Loss', color='C0')
    ax1.plot(history['val_loss'], label='Val Loss', color='C1')
    ax1.set_title("Model Loss (MSE)")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot SINR
    ax2.plot(history['train_sinr'], label='Train SINR', color='C0')
    ax2.plot(history['val_sinr'], label='Val SINR', color='C1')
    ax2.set_title("Model Quality (SINR dB)")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("dB (Higher is Better)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    return fig

In [ ]:
def calc_sinr_db(predictions, targets):
    """
    Calculates the Signal-to-Interference-Noise Ratio (SINR) in dB.
    
    Args:
        predictions (np.array): (Batch, 2, Length)
        targets (np.array): The Ground Truth Clean Signal
    Returns:
        float: The average SINR in dB across the batch.
    """
    pred_flat = predictions.reshape(predictions.shape[0], -1)
    targ_flat = targets.reshape(targets.shape[0], -1)

    # Vectorized Power Calculation (Axis 1 = across time/channels)
    signal_power = np.mean(targ_flat ** 2, axis=1)
    
    # Noise = Difference between Prediction and Target
    noise_power = np.mean((pred_flat - targ_flat) ** 2, axis=1)

    # Ratio Calculation (with epsilon for safety)
    ratio = signal_power / (noise_power + 1e-10)
    
    # Convert to dB
    sinr_db = 10 * np.log10(ratio)

    # Return scalar mean
    return float(np.mean(sinr_db))


In [ ]:
print("\n[1/4] Loading Datasets...")
train_ds = LucidRFDataset(TRAIN_X_PATH, TRAIN_Y_PATH)
val_ds = LucidRFDataset(VAL_X_PATH, VAL_Y_PATH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train Set: {len(train_ds)} | Val Set: {len(val_ds)}")

In [ ]:
print("\n[2/4] Initializing U-Net...")
model = LucidRF_UNet().to(DEVICE)

optimizer = optim.Adam(
    model.parameters(), 
    lr=LEARNING_RATE, 
    weight_decay=0.0000010202593327567395
)

# criterion = nn.MSELoss()
# criterion = ContinuousSINRLoss(min_weight=1.0, max_weight=1000.0).to(DEVICE)
criterion = PhaseAwareLoss(phase_weight=0.11106880351917545).to(DEVICE)

# Initialize Callbacks
early_stopper = EarlyStopping()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, threshold=1e-3)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

In [ ]:
start_epoch = 0
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_sinr': [], 'val_sinr': []}

if os.path.exists(checkpoint_path):
    print(f"Found checkpoint: {checkpoint_path}")
    try:
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch']
        best_val_loss = checkpoint['best_loss']
        early_stopper.best_loss = best_val_loss

        if 'history' in checkpoint:
            history = checkpoint['history']

        if 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

        if 'early_stopper_counter' in checkpoint:
            early_stopper.counter = checkpoint['early_stopper_counter']
        
        print(f"Resuming from End of Epoch {start_epoch} | Best Loss: {best_val_loss:.6f}")
            
    except Exception as e:
        print(f"Critical Error loading checkpoint: {e}")
        print("If the file is old/incompatible, delete it manually and restart.")
else:
    print("No checkpoint found. Starting fresh.")

print("\n[3/4] Starting Training...")
for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    running_sinr = 0.0
    
    # Progress bar for training
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
    for inputs, targets in pbar:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)

        loss = criterion(outputs, targets)
        # loss = criterion(outputs, targets, inputs)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.8356001021623817)
        optimizer.step()

        # bridge to pure math , detach and move to CPU for numpy
        out_np = outputs.detach().cpu().numpy()
        tgt_np = targets.detach().cpu().numpy()
        sinr = calc_sinr_db(out_np, tgt_np)
        
        running_loss += loss.item()
        running_sinr += sinr

        pbar.set_postfix({'loss': f"{loss.item():.5f}", 'sinr': f"{sinr:.1f} dB"})
    
    epoch_train_loss = running_loss / len(train_loader)
    epoch_train_sinr = running_sinr / len(train_loader)
    
    # Validation Phase
    model.eval()
    val_running_loss = 0.0
    val_running_sinr = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            outputs = model(inputs)

            val_loss = criterion(outputs, targets) 
            # val_loss = criterion(outputs, targets, inputs)

            out_np = outputs.cpu().numpy()
            tgt_np = targets.cpu().numpy()
            val_sinr_val = calc_sinr_db(out_np, tgt_np)
            
            val_running_loss += val_loss.item()
            val_running_sinr += val_sinr_val

    epoch_val_loss = val_running_loss / len(val_loader)
    epoch_val_sinr = val_running_sinr / len(val_loader)
    

    # --- CALLBACKS & LOGGING ---
    # Store history
    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['train_sinr'].append(epoch_train_sinr)
    history['val_sinr'].append(epoch_val_sinr)

    current_fig = plot_training_live_history(history)
    current_fig.savefig(train_fig, bbox_inches='tight')

    # Learning Rate Scheduler
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(epoch_val_loss)
    new_lr = optimizer.param_groups[0]['lr']
    
    print(f"Shape Check: In {inputs.shape} -> Out {outputs.shape}") # Sanity check once per epoch
    print(f"Results: Train Loss: {epoch_train_loss:.6f} | Val Loss: {epoch_val_loss:.6f} | Train SINR: {epoch_train_sinr:.1f} dB | Val SINR: {epoch_val_sinr:.1f} dB")

    log_entry = (f"Epoch {epoch+1}: Train Loss: {epoch_train_loss:.6f}, Val Loss: {epoch_val_loss:.6f}, "
                 f"Train SINR: {epoch_train_sinr:.1f}dB, Val SINR: {epoch_val_sinr:.1f}dB, LR: {new_lr:.1e}\n")

    with open(log_file, "a") as f:
        f.write(log_entry)
    
    if new_lr < old_lr:
        print(f"Learning Rate reduced: {old_lr:.1e} -> {new_lr:.1e}")

    # Checkpointing
    if epoch_val_loss < best_val_loss:
        print(f"Improvement! Saving model... ({best_val_loss:.6f} -> {epoch_val_loss:.6f})")
        best_val_loss = epoch_val_loss

        checkpoint_bundle = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_loss': best_val_loss,
            'history': history,
            'scheduler_state_dict': scheduler.state_dict(),
            'early_stopper_counter': early_stopper.counter
        }
        current_fig.savefig(best_model_fig, bbox_inches='tight')
        torch.save(checkpoint_bundle, checkpoint_path)

    plt.close(current_fig)
    # early stopping
    early_stopper(epoch_val_loss)
    if early_stopper.early_stop:
        print(f"\nEarly Stopping triggered! No improvement for {early_stopper.patience} epochs.")
        break

print("\n[4/4] Training Complete.")